# KNN-Based Electrical Grid Stability Classification

This notebook loads the UCI Electrical Grid Stability dataset, preprocesses it, selects K using training-only cross-validation, evaluates on a held-out test set, visualizes results, and predicts a new operating condition.

**Dataset source:** https://archive.ics.uci.edu/dataset/471/electrical+grid+stability+simulated+data

**Important:** The dataset is simulated. Do not use this model for real grid protection.

## 1. Imports and configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ml_backend import FEATURES, TARGET, load_dataset, prepare_data, train_evaluate, predict_one

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

ModuleNotFoundError: No module named 'ml_backend'

## 2. Load data and inspect quality

In [ ]:
df = load_dataset()
print('Dataset shape:', df.shape)
display(df.head())
print('\nData types:')
display(df.dtypes)
print('\nMissing values per column:')
display(df.isna().sum())
print('Duplicate rows:', int(df.duplicated().sum()))
print('\nClass distribution:')
display(df[TARGET].value_counts())
display(df[TARGET].value_counts(normalize=True).mul(100).round(2))
print('\nSummary statistics:')
display(df.describe(include='all').transpose())

## 3. Feature and target identification

The UCI description identifies `p1` as non-predictive. `stab` is excluded because it is a continuous stability outcome related to the classification target and could leak target information.

In [ ]:
X, y, cleaned_df = prepare_data(df, target=TARGET, features=FEATURES)
print('Predictor features:', list(X.columns))
print('Target:', TARGET)
print('Target classes:', sorted(y.unique()))
print('Cleaned observations:', len(X))

## 4. Visualize class distribution and correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y.value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Grid stability class distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Observations')
sns.heatmap(X.corr(numeric_only=True), cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Predictor correlation matrix')
plt.tight_layout()
plt.show()

## 5. Train/test split, feature selection, scaling, and K tuning

The backend uses a stratified 80:20 split (random state 42), median imputation, optional mutual-information feature selection, standard scaling, and 5-fold stratified cross-validation on training data. The held-out test set is not used to choose K.

In [ ]:
result = train_evaluate(
    X, y,
    use_feature_selection=True,
    n_features=6,
    test_size=0.20,
    random_state=42,
    k_values=[1, 3, 5, 7, 9, 11, 15, 21]
)
model = result['model']
y_test = result['y_test']
y_pred = result['predictions']
print('Training observations:', len(result['X_train']))
print('Testing observations:', len(result['X_test']))
print('Best K:', result['best_k'])
print('Mean CV accuracy:', f"{result['cv_accuracy']:.4f}")

## 6. Performance metrics

In [ ]:
print('TEST ACCURACY:', f"{result['accuracy']:.4%}")
print('WEIGHTED PRECISION:', f"{result['precision_weighted']:.4f}")
print('WEIGHTED RECALL:', f"{result['recall_weighted']:.4f}")
print('WEIGHTED F1-SCORE:', f"{result['f1_weighted']:.4f}")
if 'roc_auc' in result:
    print('ROC-AUC:', f"{result['roc_auc']:.4f}")
print('\nClassification report:')
print(result['classification_report'])
if result['accuracy'] >= 0.90:
    print('The test accuracy meets the 90% project benchmark.')
else:
    print('The test accuracy is below 90%; review features and methodology without tuning on the test set.')

## 7. Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=model.classes_, cmap='Blues', values_format='d', ax=ax)
ax.set_title('KNN confusion matrix — held-out test set')
plt.tight_layout()
plt.show()

## 8. K-selection curve

In [ ]:
cv_results = pd.DataFrame(result['search'].cv_results_)
cv_results['K'] = cv_results['param_knn__n_neighbors'].astype(int)
cv_results = cv_results.sort_values('K')
plt.figure(figsize=(8, 5))
plt.errorbar(cv_results['K'], cv_results['mean_test_score'], yerr=cv_results['std_test_score'], marker='o', capsize=4)
plt.axvline(result['best_k'], linestyle='--', label=f"Best K = {result['best_k']}")
plt.xlabel('Number of neighbours (K)')
plt.ylabel('Mean 5-fold CV accuracy')
plt.title('KNN hyperparameter selection')
plt.legend()
plt.tight_layout()
plt.show()
display(cv_results[['K', 'mean_test_score', 'std_test_score']])

## 9. Feature selection scores

In [ ]:
selector = model.named_steps['selector']
if hasattr(selector, 'scores_'):
    feature_scores = pd.DataFrame({'Feature': X.columns, 'Mutual information score': selector.scores_}).sort_values('Mutual information score', ascending=False)
    display(feature_scores)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=feature_scores, x='Mutual information score', y='Feature')
    plt.title('Feature selection scores')
    plt.tight_layout()
    plt.show()
    print('Selected features:', list(X.columns[selector.get_support()]))

## 10. Predict a new operating condition

In [ ]:
new_condition = {
    'tau1': 4.0, 'tau2': 4.0, 'tau3': 4.0, 'tau4': 4.0,
    'p2': -0.5, 'p3': -0.5, 'p4': -0.5,
    'g1': 0.5, 'g2': 0.5, 'g3': 0.5, 'g4': 0.5
}
prediction, probabilities = predict_one(model, new_condition, feature_names=FEATURES)
print('Predicted grid state:', prediction)
print('Estimated class probabilities:', probabilities)

## 11. Export test predictions and report results

Copy the measured metrics from the output above into `project_report.md` and `Project_Report.docx`. Save the notebook after executing all cells so the final submission contains actual outputs and plots.

In [ ]:
test_predictions = pd.DataFrame({'actual': y_test.to_numpy(), 'predicted': y_pred})
test_predictions.to_csv('knn_grid_test_predictions.csv', index=False)
display(test_predictions.head())